<a href="https://colab.research.google.com/github/Khushibung05/RAG/blob/main/Chromadb_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install chromadb
!pip install groq
import os
import chromadb
from groq import Groq
from google.colab import userdata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.4 MB/s eta 0:00:00


###Setup

In [17]:
chroma_client=chromadb.PersistentClient(path="./chroma_db")
collection=chroma_client.get_or_create_collection(name="ABC_Handbook")
groq_client=Groq(api_key=userdata.get('GROQ_KEY'))

In [18]:
# --- Document: ABC Employee Handbook (chunked by section) ---
handbook_chunks = [
    "### Company Structure\nABC was founded in 2024 by Mark and John. The company operates in the AI education space with teams across engineering, content, and operations.",
    "### Work Hours\nStandard work hours are 9 AM to 6 PM, Monday through Friday. Flexible start times between 8–10 AM are permitted with manager approval.",
    "### Leave Policy\nEmployees are entitled to 12 casual leaves and 6 sick leaves per year. Leaves do not carry forward to the next year.",
    "### Remote Work\nEmployees must be present in the office on Tuesdays and Wednesdays. Remote work is permitted on all other working days.",
    "### Onboarding\nNew employees are expected to complete the onboarding module within two weeks of joining. The module covers company tools, processes, and code of conduct.",
    "### Learning and Development\nABC provides access to learning platforms. Priority courses are: Python for Data Science, SQL Fundamentals, and Machine Learning Foundations.",
]



In [19]:
collection.upsert(
    documents=handbook_chunks,
    ids=[f'chunk_{i}' for i in range(len(handbook_chunks))]
)
print(f'Indexed {len(handbook_chunks)} Chunks')

Indexed 6 Chunks


###RAG Query Function

In [24]:
def ask(question:str,top_k:int=3)->str:
  #Retrieve Relevant chunks
  results=collection.query(
      query_texts=[question],
      n_results=top_k
  )
  retrieved=results["documents"][0]
  context="\n\n".join(retrieved)
  prompt=f"""Answer the user question using only the context provided below.
  if the user cannot be found in the context, reply exactly:"I am not sure."
  Do not use outside knowledge.

  context: {context}

  question: {question}"""

  response=groq_client.chat.completions.create(
      model="llama-3.3-70b-versatile",
      messages=[{"role":"user","content":prompt}]
  )
  return response.choices[0].message.content

In [25]:
questions=[
    "How many casual leaves and sick leaves do employees get?",
    "Who Founded ABC",
    "What is the comapny's revenue last quarter?"
]
for q in questions:
  print(f'Q:{q}')
  print(f'A:{ask(q)}')
  print()

Q:How many casual leaves and sick leaves do employees get?
A:Employees are entitled to 12 casual leaves and 6 sick leaves per year.

Q:Who Founded ABC
A:Mark and John.

Q:What is the comapny's revenue last quarter?
A:I am not sure.

